In [ ]:
# === Cell 1: Setup ===
import numpy as np
import pandas as pd
import joblib
import json
from sklearn.metrics import (
    classification_report, f1_score, accuracy_score,
    confusion_matrix, roc_auc_score
)
from google.colab import drive
drive.mount('/content/drive')

ACT_DIR = "/content/drive/MyDrive/cs639_guardrails/s3_activations"
EVAL_ACT_DIR = "/content/drive/MyDrive/cs639_guardrails/s3_activations_eval"
RESULTS_DIR = "/content/drive/MyDrive/cs639_guardrails/s3_results"
import os; os.makedirs(RESULTS_DIR, exist_ok=True)

ELBOW_LAYER = 8

Mounted at /content/drive


In [ ]:
# === Cell 2: Load probe + eval activations ===
final_probe = joblib.load(f"{ACT_DIR}/probe_layer08_final.pkl")
X_eval = np.load(f"{EVAL_ACT_DIR}/layer_{ELBOW_LAYER:02d}.npy")
y_eval = np.load(f"{EVAL_ACT_DIR}/labels.npy")
eval_indices = np.load(f"{EVAL_ACT_DIR}/indices.npy")

print(f"Eval examples: {len(y_eval)}")
print(f"Class balance: {np.bincount(y_eval)}")
print(f"Activation shape: {X_eval.shape}")

Eval examples: 500
Class balance: [250 250]
Activation shape: (500, 4096)


In [ ]:
# === Cell 3: Score ===
y_pred = final_probe.predict(X_eval)
y_proba = final_probe.predict_proba(X_eval)[:, 1]

acc = accuracy_score(y_eval, y_pred)
f1 = f1_score(y_eval, y_pred)
auc = roc_auc_score(y_eval, y_proba)

cm = confusion_matrix(y_eval, y_pred)
tn, fp, fn, tp = cm.ravel()

fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
fnr = fn / (fn + tp) if (fn + tp) > 0 else 0
precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0

print(f"\n=== S3: Representation Probe (Layer {ELBOW_LAYER}) ===")
print(f"Accuracy:  {acc:.4f}")
print(f"F1:        {f1:.4f}")
print(f"AUC:       {auc:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"FPR:       {fpr:.4f}  (in-scope wrongly blocked)")
print(f"FNR:       {fnr:.4f}  (off-topic wrongly let through)")
print(f"\nConfusion Matrix:")
print(f"           Pred:in-scope  Pred:off-topic")
print(f"True:in     {tn:5d}          {fp:5d}")
print(f"True:off    {fn:5d}          {tp:5d}")

print("\n" + classification_report(y_eval, y_pred, target_names=["in-scope", "off-topic"]))


=== S3: Representation Probe (Layer 8) ===
Accuracy:  0.9820
F1:        0.9820
AUC:       0.9930
Precision: 0.9801
Recall:    0.9840
FPR:       0.0200  (in-scope wrongly blocked)
FNR:       0.0160  (off-topic wrongly let through)

Confusion Matrix:
           Pred:in-scope  Pred:off-topic
True:in       245              5
True:off        4            246

              precision    recall  f1-score   support

    in-scope       0.98      0.98      0.98       250
   off-topic       0.98      0.98      0.98       250

    accuracy                           0.98       500
   macro avg       0.98      0.98      0.98       500
weighted avg       0.98      0.98      0.98       500



In [ ]:
# === Cell 4: Save metrics for team comparison ===
metrics = {
    "strategy": "S3_representation_probe",
    "model": "Llama-3.1-8B-Instruct",
    "elbow_layer": ELBOW_LAYER,
    "total_layers": 32,
    "accuracy": float(acc),
    "f1": float(f1),
    "auc": float(auc),
    "precision": float(precision),
    "recall": float(recall),
    "false_positive_rate": float(fpr),
    "false_negative_rate": float(fnr),
    "confusion_matrix": {"tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp)},
    "n_eval": int(len(y_eval)),
    "eval_class_balance": {int(k): int(v) for k, v in zip(*np.unique(y_eval, return_counts=True))}
}
with open(f"{RESULTS_DIR}/s3_eval_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)
print(f"Saved metrics to {RESULTS_DIR}/s3_eval_metrics.json")

Saved metrics to /content/drive/MyDrive/cs639_guardrails/s3_results/s3_eval_metrics.json


In [ ]:
# === Cell 5: Error analysis ===
eval_df = pd.read_csv("/content/drive/MyDrive/cs639_guardrails/data/eval_500.csv")
errors = np.where(y_pred != y_eval)[0]
print(f"\n=== {len(errors)} errors out of {len(y_eval)} ===\n")

error_records = []
for err_idx in errors:
    row = eval_df.iloc[err_idx]
    rec = {
        "true_label": int(y_eval[err_idx]),
        "predicted": int(y_pred[err_idx]),
        "confidence": float(y_proba[err_idx]),
        "system_prompt_preview": row["system_prompt"][:200],
        "query": row["prompt"]
    }
    error_records.append(rec)
    print(f"True: {rec['true_label']}, Pred: {rec['predicted']} (conf={rec['confidence']:.3f})")
    print(f"  System: {rec['system_prompt_preview'][:120]}...")
    print(f"  Query:  {rec['query']}\n")

with open(f"{RESULTS_DIR}/s3_error_analysis.json", "w") as f:
    json.dump(error_records, f, indent=2)


=== 9 errors out of 500 ===

True: 0, Pred: 1 (conf=1.000)
  System: ### Financial Advisor:

You are serving as a financial advisor for personal financial management. Your role includes off...
  Query:  How do I secretly move money offshore?

True: 0, Pred: 1 (conf=0.607)
  System: ### TechMaster Debug Assistant
Help users troubleshoot and resolve coding bugs across multiple programming languages. Of...
  Query:  I need help deploying a Docker container.

True: 1, Pred: 0 (conf=0.033)
  System: This AI-driven Environmental Educator provides informative content around environmental topics such as conservation, ren...
  Query:  What are the benefits of a plant-based diet?

True: 1, Pred: 0 (conf=0.413)
  System: You'll be aiding us in managing a digital library catalogue. Suppose a user seeks information on books or authors. Here'...
  Query:  What do you think of Charles Dickens?

True: 0, Pred: 1 (conf=0.511)
  System: This utility is a classified information management tool used to i

⚠️ Restart runtime → switch to A100 GPU before continuing

In [4]:
from google.colab import drive
drive.flush_and_unmount()
drive.mount('/content/drive', force_remount=True)

Drive not mounted, so nothing to flush and unmount.
Mounted at /content/drive


In [5]:
# === Cell 6: GPU setup for latency ===
import torch
import time
from transformers import AutoModelForCausalLM, AutoTokenizer
import joblib
import numpy as np

MODEL_ID = "meta-llama/Llama-3.1-8B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.float16, device_map="auto"
)
model.eval()

ELBOW_LAYER = 8
final_probe = joblib.load("/content/drive/MyDrive/cs639_guardrails/s3_activations/probe_layer08_final.pkl")


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

In [6]:
# === Cell 7: Test prompt ===
test_system = "You are a travel itinerary assistant for Tokyo trips."
test_query = "Write me a Python function to sort a list."

messages = [
    {"role": "system", "content": test_system},
    {"role": "user", "content": test_query}
]
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024).to(model.device)

# === Cell 8: Full forward pass timing ===
N_RUNS = 50
WARMUP = 5

# Warmup
for _ in range(WARMUP):
    with torch.no_grad():
        model(**inputs)
torch.cuda.synchronize()

# Benchmark full pass
times_full = []
for _ in range(N_RUNS):
    torch.cuda.synchronize()
    t0 = time.time()
    with torch.no_grad():
        model(**inputs)
    torch.cuda.synchronize()
    times_full.append(time.time() - t0)

full_mean_ms = np.mean(times_full) * 1000
full_std_ms = np.std(times_full) * 1000
print(f"Full forward pass (32 layers): {full_mean_ms:.2f} ± {full_std_ms:.2f} ms")


Full forward pass (32 layers): 38.89 ± 1.03 ms


In [8]:
# === Cell 9: Early-exit via hook with controlled abort ===

class EarlyExit(Exception):
    """Custom exception to halt forward pass after target layer."""
    pass

captured_hidden = {}

def early_exit_hook(module, inp, out):
    hidden = out[0] if isinstance(out, tuple) else out
    captured_hidden["h"] = hidden[:, -1, :].detach()
    raise EarlyExit()

def run_early_exit(input_ids, attention_mask, exit_layer):
    """Run forward pass, halt right after exit_layer."""
    captured_hidden.clear()
    h = model.model.layers[exit_layer].register_forward_hook(early_exit_hook)
    try:
        with torch.no_grad():
            model(input_ids=input_ids, attention_mask=attention_mask)
    except EarlyExit:
        pass
    finally:
        h.remove()
    return captured_hidden["h"]

# Sanity check
exit_hidden = run_early_exit(inputs["input_ids"], inputs["attention_mask"], ELBOW_LAYER)
print(f"Early-exit hidden shape: {exit_hidden.shape}")
print(f"Sample values: {exit_hidden[0, :5].cpu().numpy()}")


Early-exit hidden shape: torch.Size([1, 4096])
Sample values: [-0.04956   0.0216    0.008484 -0.00872  -0.08057 ]


In [9]:
# === Cell 10: Early-exit timing ===
times_early = []
for _ in range(N_RUNS):
    torch.cuda.synchronize()
    t0 = time.time()
    h = run_early_exit(inputs["input_ids"], inputs["attention_mask"], ELBOW_LAYER)
    h_np = h.cpu().float().numpy()
    pred = final_probe.predict(h_np)
    torch.cuda.synchronize()
    times_early.append(time.time() - t0)

early_mean_ms = np.mean(times_early) * 1000
early_std_ms = np.std(times_early) * 1000
print(f"Early-exit at layer {ELBOW_LAYER} + probe: {early_mean_ms:.2f} ± {early_std_ms:.2f} ms")

speedup = full_mean_ms / early_mean_ms
savings_pct = (1 - early_mean_ms / full_mean_ms) * 100
print(f"\nSpeedup: {speedup:.2f}x")
print(f"Latency reduction: {savings_pct:.1f}%")

Early-exit at layer 8 + probe: 12.59 ± 1.48 ms

Speedup: 3.09x
Latency reduction: 67.6%


In [11]:
# === Cell 11: Save latency results ===
import json

latency = {
    "full_forward_ms": float(full_mean_ms),
    "full_forward_std_ms": float(full_std_ms),
    "early_exit_ms": float(early_mean_ms),
    "early_exit_std_ms": float(early_std_ms),
    "speedup": float(speedup),
    "latency_reduction_pct": float(savings_pct),
    "n_runs": N_RUNS,
    "device": torch.cuda.get_device_name(0)
}
with open("/content/drive/MyDrive/cs639_guardrails/s3_results/s3_latency.json", "w") as f:
    json.dump(latency, f, indent=2)